# Data cleaning: merge WT, EMD attendance, and bed occupancy

This notebook combines:
- `WT for Admission to Ward.csv`
- `Attendances at EMD.csv`
- `Bed Occupancy Rate.csv`

It aligns them to a **common starting date** and adds `Is_Holiday` using `public holidays.csv`.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Files (assumed in the same folder as this notebook)
BASE_DIR = Path.cwd()

WT_PATH = BASE_DIR / "WT for Admission to Ward.csv"
EMD_PATH = BASE_DIR / "Attendances at EMD.csv"
BOR_PATH = BASE_DIR / "Bed Occupancy Rate.csv"
HOLIDAYS_PATH = BASE_DIR / "public holidays.csv"

# Final output (long format)
OUTPUT_PATH = BASE_DIR / "combined_hospital_daily_metrics.csv"

print("Input files:")
print("-", WT_PATH)
print("-", EMD_PATH)
print("-", BOR_PATH)
print("-", HOLIDAYS_PATH)
print("Output:")
print("-", OUTPUT_PATH)


Input files:
- c:\Users\corva\Downloads\Machine Learning\WT for Admission to Ward.csv
- c:\Users\corva\Downloads\Machine Learning\Attendances at EMD.csv
- c:\Users\corva\Downloads\Machine Learning\Bed Occupancy Rate.csv
- c:\Users\corva\Downloads\Machine Learning\public holidays.csv
Output:
- c:\Users\corva\Downloads\Machine Learning\combined_hospital_daily_metrics.csv


In [14]:
def _parse_ddmmyy_from_mixed_date(series: pd.Series) -> pd.Series:
    """Parse dates like 'Sun, 01/01/23' into datetime64[ns]."""
    extracted = series.astype(str).str.extract(r"(\d{2}/\d{2}/\d{2})", expand=False)
    return pd.to_datetime(extracted, format="%d/%m/%y", errors="coerce")


def _prefix_columns(df: pd.DataFrame, prefix: str, date_col: str = "Date") -> pd.DataFrame:
    df = df.copy()
    rename_map = {c: f"{prefix}{c}" for c in df.columns if c != date_col}
    return df.rename(columns=rename_map)


In [15]:
# Load datasets
wt_raw = pd.read_csv(WT_PATH)
emd_raw = pd.read_csv(EMD_PATH)
bor_raw = pd.read_csv(BOR_PATH)

# Parse/standardize date columns
wt = wt_raw.copy()
wt["Date"] = _parse_ddmmyy_from_mixed_date(wt["Date"])

emd = emd_raw.copy()
emd["Date"] = _parse_ddmmyy_from_mixed_date(emd["Date"])

bor = bor_raw.copy()
bor["Date"] = pd.to_datetime(bor["Date"], format="%d/%m/%Y", errors="coerce")

# Drop rows where Date could not be parsed
wt = wt.dropna(subset=["Date"]).reset_index(drop=True)
emd = emd.dropna(subset=["Date"]).reset_index(drop=True)
bor = bor.dropna(subset=["Date"]).reset_index(drop=True)

# Remove completely empty unnamed columns that sometimes appear
wt = wt.loc[:, ~wt.columns.astype(str).str.match(r"^Unnamed")]
emd = emd.loc[:, ~emd.columns.astype(str).str.match(r"^Unnamed")]
bor = bor.loc[:, ~bor.columns.astype(str).str.match(r"^Unnamed")]

# Prefix to avoid collisions (same hospital names across files)
wt = _prefix_columns(wt, prefix="WT_")
emd = _prefix_columns(emd, prefix="EMD_")
bor = _prefix_columns(bor, prefix="BOR_")

print("Parsed date ranges:")
print("WT :", wt["Date"].min().date(), "->", wt["Date"].max().date(), "rows=", len(wt))
print("EMD:", emd["Date"].min().date(), "->", emd["Date"].max().date(), "rows=", len(emd))
print("BOR:", bor["Date"].min().date(), "->", bor["Date"].max().date(), "rows=", len(bor))


Parsed date ranges:
WT : 2023-01-01 -> 2026-01-10 rows= 1106
EMD: 2023-01-01 -> 2026-01-10 rows= 1106
BOR: 2018-01-01 -> 2026-01-10 rows= 2932


In [16]:
# Common starting date (latest of the three starts)
common_start = max(wt["Date"].min(), emd["Date"].min(), bor["Date"].min())
print("Common starting date:", common_start.date())

# Filter all datasets to the common start
wt_f = wt[wt["Date"] >= common_start].copy()
emd_f = emd[emd["Date"] >= common_start].copy()
bor_f = bor[bor["Date"] >= common_start].copy()

# Inner merge on Date so the final dataset only contains dates present in all three
combined = wt_f.merge(emd_f, on="Date", how="inner").merge(bor_f, on="Date", how="inner")
combined = combined.sort_values("Date").reset_index(drop=True)

print("Combined rows:", len(combined))
print("Combined date range:", combined["Date"].min().date(), "->", combined["Date"].max().date())
print("Combined columns:", combined.shape[1])


Common starting date: 2023-01-01
Combined rows: 1106
Combined date range: 2023-01-01 -> 2026-01-10
Combined columns: 29


In [17]:
# Load public holidays and create Is_Holiday (0/1)
# Note: this CSV has some multiline quotes in holiday names; engine='python' handles it more robustly.
holidays = pd.read_csv(HOLIDAYS_PATH, engine="python")
holidays["date"] = pd.to_datetime(holidays["date"], format="%d/%m/%Y", errors="coerce")
holiday_dates = set(holidays.dropna(subset=["date"])["date"].dt.date.tolist())

combined["Is_Holiday"] = combined["Date"].dt.date.map(lambda d: 1 if d in holiday_dates else 0).astype(int)

print("Holiday rows in combined:", int(combined["Is_Holiday"].sum()))
combined.head()


Holiday rows in combined: 36


,Date,WT_AH,WT_CGH,WT_KTPH,WT_NTFGH,WT_NUH(A),WT_SGH,WT_SKH,WT_TTSH,WT_WH,...,BOR_AH,BOR_CGH,BOR_KTPH,BOR_NTFGH,BOR_NUH(A),BOR_SGH,BOR_SKH,BOR_TTSH,BOR_WH,Is_Holiday
0,2023-01-01,1.6,7.7,2.9,7.9,2.1,1.3,3.0,3.8,NaN,...,80.5%,89.6%,96.4%,91.1%,78.3%,79.3%,86.0%,91.2%,NaN,1
1,2023-01-02,2.0,12.7,12.2,4.2,2.5,2.5,2.7,4.4,NaN,...,84.8%,90.6%,97.9%,92.0%,82.8%,84.0%,87.5%,95.9%,NaN,0
2,2023-01-03,2.1,16.0,12.9,22.8,4.0,9.3,5.0,4.9,NaN,...,92.9%,89.8%,98.4%,91.9%,86.7%,88.4%,88.6%,96.7%,NaN,0
3,2023-01-04,2.5,16.3,21.4,14.7,8.2,11.6,10.3,6.9,NaN,...,91.1%,87.7%,96.4%,89.9%,86.6%,90.2%,89.5%,97.7%,NaN,0
4,2023-01-05,2.4,21.5,20.7,18.0,5.2,11.7,7.9,5.0,NaN,...,92.9%,89.5%,95.1%,89.2%,88.2%,90.8%,90.2%,96.9%,NaN,0


In [18]:
# Reshape to the required long format:
# Date, Day, Hospital, Waiting Time, Attendance, BOR, Is_Holiday

# Hospitals are the intersection of the three sources
wt_h = {c.removeprefix("WT_") for c in combined.columns if c.startswith("WT_")}
emd_h = {c.removeprefix("EMD_") for c in combined.columns if c.startswith("EMD_")}
bor_h = {c.removeprefix("BOR_") for c in combined.columns if c.startswith("BOR_") and c not in {"BOR_Years"}}

hospitals = sorted(wt_h & emd_h & bor_h)
print("Hospitals:", hospitals)

rows = []
for h in hospitals:
    rows.append(
        combined[["Date", "Is_Holiday", f"WT_{h}", f"EMD_{h}", f"BOR_{h}"]]
        .rename(
            columns={
                f"WT_{h}": "Waiting Time",
                f"EMD_{h}": "Attendance",
                f"BOR_{h}": "BOR",
            }
        )
        .assign(Hospital=h)
    )

final_df = pd.concat(rows, ignore_index=True)

# ADDED: Extract the day of the week name
final_df["Day"] = final_df["Date"].dt.day_name()

# Clean BOR: convert '80.5%' -> 80.5 (float). Keep NaNs if missing.
final_df["BOR"] = (
    final_df["BOR"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .replace({"nan": np.nan, "None": np.nan, "": np.nan})
)
final_df["BOR"] = pd.to_numeric(final_df["BOR"], errors="coerce")

# UPDATED: Reorder columns to include 'Day' beside 'Date'
final_df = final_df[["Date", "Day", "Hospital", "Waiting Time", "Attendance", "BOR", "Is_Holiday"]]
final_df = final_df.sort_values(["Date", "Hospital"]).reset_index(drop=True)

# Save to CSV
final_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

final_df.head()

Hospitals: ['AH', 'CGH', 'KTPH', 'NTFGH', 'NUH(A)', 'SGH', 'SKH', 'TTSH', 'WH']
Saved: c:\Users\corva\Downloads\Machine Learning\combined_hospital_daily_metrics.csv


,Date,Day,Hospital,Waiting Time,Attendance,BOR,Is_Holiday
0,2023-01-01,Sunday,AH,1.6,64.0,80.5,1
1,2023-01-01,Sunday,CGH,7.7,351.0,89.6,1
2,2023-01-01,Sunday,KTPH,2.9,286.0,96.4,1
3,2023-01-01,Sunday,NTFGH,7.9,252.0,91.1,1
4,2023-01-01,Sunday,NUH(A),2.1,257.0,78.3,1
